# LoRA pilot on dev (Colab): does the distilled 8B write better NARS rules?

The same 30 random **dev** questions per dataset, the same 70B facts and ONA.py options; only the rule writer changes.

| Row | Rule writer | Served by |
|---|---|---|
| NARS + DeepSeek-V3 | the teacher | DeepInfra |
| NARS + 8B | Llama-3.1-8B, untrained | vLLM on this GPU |
| NARS + 8B + LoRA | the same 8B with your adapter | vLLM on this GPU |
| 8B direct | no NARS: the 8B answers from the facts | vLLM on this GPU |

The three 8B rows share one vLLM server, so they differ only by the adapter.

**Before you run:**
1. Runtime → Change runtime type → **A100**.
2. Add your DeepInfra key as a Colab secret: the 🔑 icon on the left → **Add new secret** → name `DEEPINFRA_API_KEY` → turn on *Notebook access*.
3. Runtime → **Run all**. Setup takes about 10 minutes, the pilot about an hour.

Answers are saved to Drive as they finish (`MyDrive/nars_lora/pilot_dev_30q_seed0/`): if the session stops, run all again and only the missing answers run. The last cell releases the GPU.

In [ ]:
%%capture
# The code, ONA (the version used for every result so far) and vLLM
!git clone -q https://github.com/MinaGabriel/llm-vs-reasoning.git /content/llm-vs-reasoning || git -C /content/llm-vs-reasoning pull -q
!git clone -q https://github.com/opennars/OpenNARS-for-Applications.git /content/ONA || true
!cd /content/ONA && git checkout -q dc4efd0abd520cdb79bf53bfa3c285ebb24f2e8a && sh build.sh
!pip install -q vllm prettytable

In [ ]:
import os, subprocess, sys, time, urllib.request
from google.colab import drive, userdata

os.chdir("/content/llm-vs-reasoning")
sys.path.insert(0, "/content/llm-vs-reasoning")
os.environ["DEEPINFRA_API_KEY"] = userdata.get("DEEPINFRA_API_KEY")
assert os.path.exists("/content/ONA/NAR"), "ONA did not build: run the first cell without %%capture to see why"

drive.mount("/content/drive")
ADAPTER = "/content/drive/MyDrive/nars_lora/llama31_8b_nars_r16/adapter"
assert os.path.exists(f"{ADAPTER}/adapter_model.safetensors"), f"no adapter in {ADAPTER}"

# vLLM: the base 8B (Meta's weights, ungated copy: the one the adapter was trained on) plus the adapter
server = subprocess.Popen(
    ["vllm", "serve", "unsloth/Meta-Llama-3.1-8B-Instruct", "--served-model-name", "llama-3.1-8b-instruct",
     "--dtype", "bfloat16", "--enable-lora", "--lora-modules", f"nars-lora={ADAPTER}", "--max-lora-rank", "16",
     "--max-model-len", "16384", "--gpu-memory-utilization", "0.90", "--port", "8000"],
    stdout=open("/content/vllm.log", "w"), stderr=subprocess.STDOUT)
for minute in range(20):                          # the first start downloads 16 GB of weights
    time.sleep(30)
    try:
        urllib.request.urlopen("http://localhost:8000/health", timeout=5)
        print("vLLM is up")
        break
    except Exception:
        if server.poll() is not None:
            print(open("/content/vllm.log").read()[-3000:])
            raise SystemExit("vLLM stopped: see the log above")
        print(f"starting vLLM... {(minute + 1) / 2:.1f} min")
else:
    raise SystemExit("vLLM did not start in 10 minutes: see /content/vllm.log")

In [ ]:
# ── Settings ─────────────────────────────────────────────────────────────────────────────────────
import json, math, pickle, random, re, unicodedata
from concurrent.futures import ThreadPoolExecutor, as_completed

from openai import OpenAI
from prettytable import PrettyTable

import ONA
from keys import deepinfra_key

CLIENTS = {
    "deepinfra": OpenAI(api_key=deepinfra_key(), base_url="https://api.deepinfra.com/v1/openai", max_retries=2, timeout=90),
    "vllm": OpenAI(api_key="none", base_url="http://localhost:8000/v1", max_retries=2, timeout=300),
}
SYSTEMS = {                                       # row → (server, rule writer); direct = no NARS
    "NARS + DeepSeek-V3": ("deepinfra", "deepseek-ai/DeepSeek-V3"),
    "NARS + 8B": ("vllm", "llama-3.1-8b-instruct"),
    "NARS + 8B + LoRA": ("vllm", "nars-lora"),
    "8B direct": ("vllm", "llama-3.1-8b-instruct"),
}
DIRECT = {"8B direct"}
DATASETS = ["MuSiQue", "HotpotQA", "2Wiki"]       # 2Wiki was held out of training: a transfer test
N_QUESTIONS = 30
SEED = 0
WORKERS = 12
ONA_OPTIONS = {"plan": True, "votes": 3, "merge_relations": True}
ONA_PATH = "/content/ONA/NAR"
JUDGE_MODEL = "Qwen/Qwen2.5-72B-Instruct"
RUN_DIR = f"/content/drive/MyDrive/nars_lora/pilot_dev_{N_QUESTIONS}q_seed{SEED}"
os.makedirs(RUN_DIR, exist_ok=True)


def load_prices():
    try:
        with urllib.request.urlopen("https://api.deepinfra.com/models/list", timeout=20) as response:
            return {m["model_name"]: (m["pricing"]["cents_per_input_token"] * 1e4, m["pricing"]["cents_per_output_token"] * 1e4)
                    for m in json.load(response) if (m.get("pricing") or {}).get("type") == "tokens"}
    except Exception:
        return {}


PRICES = load_prices()


def cost(input_tokens, output_tokens, model):     # vLLM rows cost GPU time only: $0 here
    price = PRICES.get(model, (0, 0))
    return (input_tokens * price[0] + output_tokens * price[1]) / 1_000_000


for server_name, model in {*SYSTEMS.values(), ("deepinfra", JUDGE_MODEL)}:
    served = CLIENTS[server_name].chat.completions.create(
        model=model, max_tokens=1, messages=[{"role": "user", "content": "Hi"}]).model
    assert served == model, f"{model} is served as {served}"
print("all models answer")

In [ ]:
# ── Scoring: the same three metrics, judge and direct prompt as 2_hop_test.ipynb ─────────────────
def normalize(text):
    text = unicodedata.normalize("NFKD", str(text)).encode("ascii", "ignore").decode().casefold()
    return re.sub(r"[^a-z0-9]", "", re.sub(r"\b(a|an|the)\b", " ", text))


def golds(q):
    return [q["answer"], *(q.get("gold_answers") or [])]


def exact(answer, g):
    return any(normalize(answer or "") == normalize(x) for x in g)


def lenient(answer, g):
    a = normalize(answer or "")
    for x in map(normalize, g):
        short, long_ = sorted((a, x), key=len)
        if short and short in long_ and 2 * len(short) >= len(long_):
            return True
    return False


JUDGE_PROMPT = """You grade answers to a question. Be strict.

Question: {question}
Gold answer: {gold}
Model answer: {answer}

The model answer is correct ONLY if it names the same entity, date, number or place as the gold answer.
Different wording, a full name, or extra detail around the same thing is fine:
  "Philip Mountbatten" for "Prince Philip" -> yes
  "12,000 people" for "12,000" -> yes
  "4 June 1990" for "1990" -> yes
It is WRONG if it names something else, even something related: a part of the gold answer, a larger thing that
contains it, a person connected to it, or a vaguer or partial answer:
  "Battle of Hastings" for "Norman Conquest" -> no   (a part, not the same thing)
  "Henry VIII" for "Edward VI" -> no                 (a related person)
  "Europe" for "France" -> no                        (broader)
  "200 km" for "200 km south of Paris" -> no         (partial)
  "The text does not say" -> no                      (no answer)
Reply with only yes or no."""


def metrics(q, answer):
    g = golds(q)
    result = {"exact": exact(answer, g), "lenient": lenient(answer, g), "judge_cost": 0.0}
    result["judge"] = result["lenient"]
    if answer and not result["lenient"]:
        try:
            response = CLIENTS["deepinfra"].chat.completions.create(model=JUDGE_MODEL, temperature=0, max_tokens=3, messages=[{
                "role": "user", "content": JUDGE_PROMPT.format(question=q["question"], gold=q["answer"], answer=answer)}])
            result["judge"] = (response.choices[0].message.content or "").strip().lower().startswith("yes")
            result["judge_cost"] = cost(response.usage.prompt_tokens or 0, response.usage.completion_tokens or 0, JUDGE_MODEL)
        except Exception as error:
            print(f"   ! judge failed ({type(error).__name__}), using the lenient match")
    return result


def facts(q):
    return [p["facts"] for p in q["supporting"] if p["facts"]]


DIRECT_PROMPT = """Answer the question using only the facts below. Return only the short answer, without explanation.

Facts:
{facts}

Question: {question}
Answer:"""


def run_one(q, index, system):
    server_name, model = SYSTEMS[system]
    client = CLIENTS[server_name]
    record = {"index": index, "question": q["question"], "gold": q["answer"], "error": None}
    if system in DIRECT:
        try:
            response = client.chat.completions.create(model=model, temperature=0, max_tokens=64, messages=[{
                "role": "user", "content": DIRECT_PROMPT.format(
                    facts="\n".join(f"- {f}" for p in facts(q) for f in p), question=q["question"])}])
            answer = (response.choices[0].message.content or "").strip().split("\n")[0]  # first line only, as 2_hop_test
            record["cost"] = cost(response.usage.prompt_tokens or 0, response.usage.completion_tokens or 0, model)
        except Exception as error:
            answer, record["error"], record["cost"] = None, f"{type(error).__name__}: {error}", 0.0
        record.update(answer=answer, has_rules=None)
    else:
        ona = ONA.OnaReasoner(client, model=model, **ONA_OPTIONS)
        answer = None
        try:
            answer = ona.solve(facts(q), q["question"], ONA_PATH)
        except Exception as error:
            record["error"] = f"{type(error).__name__}: {error}"
        record.update(answer=answer, rules=ona.last_rules, has_rules=bool(ona.last_rules),
                      auto_repairs=dict(ona.auto_repairs), votes=[s["answer"] for s in ona.samples],
                      cost=cost(ona.usage["input_tokens"], ona.usage["output_tokens"], model))
    record.update(metrics(q, record["answer"]))
    return record


def path(dataset, system):
    return f"{RUN_DIR}/{dataset}__{re.sub(r'[^A-Za-z0-9]+', '_', system).strip('_')}.jsonl"


def load(dataset, system):
    if not os.path.exists(path(dataset, system)):
        return {}
    with open(path(dataset, system)) as f:
        return {r["index"]: r for r in map(json.loads, f) if not r.get("error")}   # errors are retried

In [ ]:
# ── Run everything that is missing ───────────────────────────────────────────────────────────────
QUESTIONS, INDICES = {}, {}
for dataset in DATASETS:
    with open(f"runs/{dataset}_dev_entity.pkl", "rb") as f:
        QUESTIONS[dataset] = pickle.load(f)
    pool = [i for i, q in enumerate(QUESTIONS[dataset]) if q["hops"] == 2]
    INDICES[dataset] = sorted(random.Random(SEED).sample(pool, min(N_QUESTIONS, len(pool))))

jobs = [(d, s, i) for d in DATASETS for s in SYSTEMS for i in INDICES[d] if i not in load(d, s)]
print(f"{len(jobs)} answers to run ({sum(len(v) for v in INDICES.values()) * len(SYSTEMS) - len(jobs)} already saved)")
start, spent = time.time(), 0.0
with ThreadPoolExecutor(WORKERS) as executor:
    futures = {executor.submit(run_one, QUESTIONS[d][i], i, s): (d, s) for d, s, i in jobs}
    for done, future in enumerate(as_completed(futures), 1):
        dataset, system = futures[future]
        r = future.result()
        with open(path(dataset, system), "a") as f:
            f.write(json.dumps(r, ensure_ascii=False, default=str) + "\n")
        spent += r["cost"] + r["judge_cost"]
        mark = "!" if r["error"] else "✓" if r["judge"] else "✗"
        print(f"[{done:>3}/{len(jobs)}] {dataset:<8} {system:<18} {mark} {str(r['answer'] or '-')[:28]:<28} "
              f"gold {r['gold'][:24]:<24} ${spent:.3f} {time.time() - start:.0f}s")
errors = sum(1 for d in DATASETS for s in SYSTEMS for i in INDICES[d] if i not in load(d, s))
print(f"\ndone: ${spent:.3f} this run" + (f" · {errors} answers failed with an error: run this cell again" if errors else ""))

## Report

In [ ]:
def mcnemar(a, b):
    """Exact two-sided McNemar p-value from the questions only A got right (a) and only B got right (b)."""
    n, k = a + b, min(a, b)
    return 1.0 if n == 0 else min(1.0, 2 * sum(math.comb(n, i) for i in range(k + 1)) / 2 ** n)


results = {(d, s): load(d, s) for d in DATASETS for s in SYSTEMS}
lines = []
table = PrettyTable(["System", *DATASETS, "All (judge)", "no rules", "cost"])
for s in SYSTEMS:
    row, total, n_all = [s], 0, 0
    for d in DATASETS:
        rs = results[d, s]
        row.append(" / ".join(str(sum(r[m] for r in rs.values())) for m in ("exact", "lenient", "judge")) + f"  of {len(rs)}")
        total += sum(r["judge"] for r in rs.values())
        n_all += len(rs)
    all_rs = [r for d in DATASETS for r in results[d, s].values()]
    row += [f"{total}/{n_all} = {100 * total / max(n_all, 1):.0f}%", "-" if s in DIRECT else sum(r["has_rules"] is False for r in all_rs),
            f"${sum(r['cost'] + r['judge_cost'] for r in all_rs):.3f}"]
    table.add_row(row)
lines += ["DEV PILOT · exact / lenient / judge per dataset", str(table)]

pairs = [("NARS + 8B + LoRA", "NARS + 8B"), ("NARS + 8B + LoRA", "NARS + DeepSeek-V3"), ("NARS + 8B + LoRA", "8B direct")]
paired = PrettyTable(["A vs B", *DATASETS, "All", "p (All)"])
for a, b in pairs:
    row, wa, wb = [f"{a} vs {b}"], 0, 0
    for d in DATASETS:
        ra, rb = results[d, a], results[d, b]
        common = ra.keys() & rb.keys()
        x = sum(ra[i]["judge"] and not rb[i]["judge"] for i in common)
        y = sum(rb[i]["judge"] and not ra[i]["judge"] for i in common)
        row.append(f"+{x} / -{y}")
        wa, wb = wa + x, wb + y
    paired.add_row(row + [f"+{wa} / -{wb}", f"{mcnemar(wa, wb):.3f}"])
lines += ["", "PAIRED (judge, same questions): +questions only A got right / -questions only B got right, exact McNemar p",
          str(paired), "30 questions per dataset only show the direction: decide on the full dev set, report on test."]
print("\n".join(lines))
with open(f"{RUN_DIR}/report.txt", "w") as f:
    f.write("\n".join(lines) + "\n")
print("\nsaved to", f"{RUN_DIR}/report.txt")

## Done: release the GPU

In [ ]:
from google.colab import runtime
server.terminate()
drive.flush_and_unmount()
runtime.unassign()